# Import Dependencies

In [7]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm.notebook import tqdm

print("Libraries imported successfully.")

Libraries imported successfully.


# HighD Dataset Load

In [8]:
# Path to the HighD dataset on Kaggle
DATA_PATH = "/kaggle/input/highd-dataset-v1-0/data"

# Output path for the final features file
OUTPUT_PATH = "/kaggle/working/"

# Define the prediction horizon for intention (in seconds)
# As per your description, this is the "target"
PREDICTION_HORIZON_S = 5.0

# Get a sorted list of all file paths
track_files = sorted(glob.glob(os.path.join(DATA_PATH, "*_tracks.csv")))
rec_meta_files = sorted(glob.glob(os.path.join(DATA_PATH, "*_recordingMeta.csv")))

print(f"Found {len(track_files)} recordings to process.")

Found 60 recordings to process.


# Helper Function for Contexual Features

In [9]:
def compute_ttc_preceding(p_ego, v_ego, p_leader, v_leader, leader_length):
    """
    Calculates TTC for a preceding vehicle.
    p = position (x), v = velocity (xVelocity), length = vehicle length (height)
    """
    delta_x = p_leader - p_ego - leader_length
    delta_v = v_ego - v_leader
    
    # Initialize all TTCs to a safe, high value
    ttc = np.full_like(delta_x, 9999.0)
    
    # Calculate TTC only where follower is faster (delta_v > 0) and is behind (delta_x > 0)
    mask = (delta_v > 0) & (delta_x > 0)
    ttc[mask] = delta_x[mask] / delta_v[mask]
    
    # Fill any NaNs (from missing vehicles) with the safe value
    return np.nan_to_num(ttc, nan=9999.0)

def compute_ttc_following(p_follower, v_follower, p_ego, v_ego, ego_length):
    """
    Calculates TTC for a following vehicle.
    Note the swapped arguments for ego/follower.
    """
    delta_x = p_ego - p_follower - ego_length
    delta_v = v_follower - v_ego
    
    # Initialize all TTCs to a safe, high value
    ttc = np.full_like(delta_x, 9999.0)
    
    # Calculate TTC only where follower is faster (delta_v > 0) and is behind (delta_x > 0)
    mask = (delta_v > 0) & (delta_x > 0)
    ttc[mask] = delta_x[mask] / delta_v[mask]
    
    # Fill any NaNs (from missing vehicles) with the safe value
    return np.nan_to_num(ttc, nan=9999.0)

print("TTC helper functions defined.")

TTC helper functions defined.


# Helper Function for Lane Change Intention

In [10]:
def compute_intention(group, horizon_frames):
    """
    Computes the future lane change intention for a single vehicle's track.
    This function is intended to be used with pandas.groupby().apply().
    """
    # Get the laneId N frames in the future. Fill end-of-track NaNs.
    group['future_laneId'] = group['laneId'].shift(-horizon_frames)
    
    # Default intention is Lane Keep (LK)
    group['intention'] = 'LK'
    
    # In highD, a higher laneId is always "left" (e.g., lane 1 -> 2, or lane 4 -> 5)
    group.loc[group['future_laneId'] > group['laneId'], 'intention'] = 'LLC'
    
    # A lower laneId is always "right"
    group.loc[group['future_laneId'] < group['laneId'], 'intention'] = 'RLC'
    
    # For frames at the end of a track where we can't see the future,
    # we will back-fill the last known intention.
    # If the track is too short, it defaults to 'LK'.
    group['intention'] = group['intention'].bfill().fillna('LK')
    
    return group

print("Intention helper function defined.")

Intention helper function defined.


# Feature Extraction on Recordings

In [ ]:
import warnings
from pandas.errors import PerformanceWarning

# --- Suppress PerformanceWarnings (for the pd.merge operations) ---
warnings.filterwarnings('ignore', category=PerformanceWarning)

all_features_list = []

print(f"Starting feature extraction for {len(track_files)} recordings...")

for i in tqdm(range(len(track_files))):
    
    # --- 1. Load Data ---
    track_path = track_files[i]
    meta_path = rec_meta_files[i]
    recording_id = int(os.path.basename(track_path).split('_')[0])
    
    track_df = pd.read_csv(track_path)
    rec_meta_df = pd.read_csv(meta_path)
    
    frame_rate = rec_meta_df['frameRate'].iloc[0]
    horizon_frames = int(PREDICTION_HORIZON_S * frame_rate)
    
    
    # --- 2. Calculate Intention (Target Label) ---
    track_df = track_df.sort_values(by=['id', 'frame'])
    track_df = track_df.groupby('id', group_keys=False).apply(compute_intention, horizon_frames=horizon_frames)
    
    
    # --- 3. Calculate All TTCs as Separate Variables ---
    neighbor_df = track_df[['frame', 'id', 'x', 'xVelocity', 'height']].rename(columns={
        'id': 'neighbor_id', 
        'x': 'neighbor_x', 
        'xVelocity': 'neighbor_v', 
        'height': 'neighbor_length'
    })

    # (a) TTC_preceding
    ttc_preceding = track_df['ttc'].replace(-1, 9999.0).fillna(9999.0)

    # (b) TTC_left_preceding
    merged_lp = pd.merge(track_df, neighbor_df, left_on=['frame', 'leftPrecedingId'], right_on=['frame', 'neighbor_id'], how='left')
    ttc_left_preceding = compute_ttc_preceding(merged_lp['x'], merged_lp['xVelocity'], merged_lp['neighbor_x'], merged_lp['neighbor_v'], merged_lp['neighbor_length'])

    # (c) TTC_left_following
    merged_lf = pd.merge(track_df, neighbor_df, left_on=['frame', 'leftFollowingId'], right_on=['frame', 'neighbor_id'], how='left')
    ttc_left_following = compute_ttc_following(merged_lf['neighbor_x'], merged_lf['neighbor_v'], merged_lf['x'], merged_lf['xVelocity'], merged_lf['height'])

    # (d) TTC_right_preceding
    merged_rp = pd.merge(track_df, neighbor_df, left_on=['frame', 'rightPrecedingId'], right_on=['frame', 'neighbor_id'], how='left')
    ttc_right_preceding = compute_ttc_preceding(merged_rp['x'], merged_rp['xVelocity'], merged_rp['neighbor_x'], merged_rp['neighbor_v'], merged_rp['neighbor_length'])

    # (e) TTC_right_following
    merged_rf = pd.merge(track_df, neighbor_df, left_on=['frame', 'rightFollowingId'], right_on=['frame', 'neighbor_id'], how='left')
    ttc_right_following = compute_ttc_following(merged_rf['neighbor_x'], merged_rf['neighbor_v'], merged_rf['x'], merged_rf['xVelocity'], merged_rf['height'])
    

    # --- 4. Assemble Final DataFrame Efficiently using .assign() ---
    
    final_features_df = track_df.assign(
        recording_id = recording_id,
        lateral_velocity = track_df['yVelocity'],
        lateral_acceleration = track_df['yAcceleration'],
        TTC_preceding = ttc_preceding,
        TTC_left_preceding = ttc_left_preceding,
        TTC_left_following = ttc_left_following,
        TTC_right_preceding = ttc_right_preceding,
        TTC_right_following = ttc_right_following
    )
    
    # --- 5. Select Only the Columns We Need ---
    final_cols_to_keep = [
        'recording_id', 
        'frame', 
        'id', 
        'lateral_velocity',
        'lateral_acceleration',
        'intention',
        'TTC_preceding',
        'TTC_left_preceding',
        'TTC_left_following',
        'TTC_right_preceding',
        'TTC_right_following'
    ]
    
    # This selection creates the final, clean DataFrame for this loop
    features_to_append = final_features_df[final_cols_to_keep]
    
    all_features_list.append(features_to_append)

print("All recordings processed.")

Starting feature extraction for 60 recordings...


  0%|          | 0/60 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: Ru

# Concatenation All Features into a Single Dataframe

In [12]:
print("Concatenating all features into a single DataFrame...")
full_features_df = pd.concat(all_features_list, ignore_index=True)

# Define the final output file path
output_filename = os.path.join(OUTPUT_PATH, "highd_extracted_features.csv")

# Save to CSV
full_features_df.to_csv(output_filename, index=False)

print(f"Feature extraction complete!")
print(f"Final DataFrame shape: {full_features_df.shape}")
print(f"File saved to: {output_filename}")

# Display the first few rows of the final dataset
print("\n--- Sample of Final Data ---")
print(full_features_df.head())

Concatenating all features into a single DataFrame...
Feature extraction complete!
Final DataFrame shape: (39725708, 11)
File saved to: /kaggle/working/highd_extracted_features.csv

--- Sample of Final Data ---
   recording_id  frame  id  lateral_velocity  lateral_acceleration intention  \
0             1      1   1              0.00                   0.0        LK   
1             1      2   1              0.00                   0.0        LK   
2             1      3   1              0.00                   0.0        LK   
3             1      4   1              0.00                   0.0        LK   
4             1      5   1              0.01                   0.0        LK   

   TTC_preceding  TTC_left_preceding  TTC_left_following  TTC_right_preceding  \
0            0.0              9999.0              9999.0               9999.0   
1            0.0              9999.0              9999.0               9999.0   
2            0.0              9999.0              9999.0         

# Load Additional Dependencies

In [5]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import warnings
import os
import glob
from tqdm import tqdm

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

Libraries imported successfully.


In [7]:
# --- Configuration ---
# !!! IMPORTANT: Update this path to match your Kaggle dataset folder !!!
DATA_DIR = '/kaggle/input/highd-dataset-v1-0/data' # e.g., /kaggle/input/highd-dataset-files/

# --- Find all files ---
# We use glob to find all 60 files and sort them
track_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_tracks.csv')))
meta_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_tracksMeta.csv')))

print(f"Found {len(track_files)} track files.")
print(f"Found {len(meta_files)} meta files.")

if not track_files:
    print(f"\n--- ERROR ---")
    print(f"No files found in '{DATA_DIR}'.")
    print("Please double-check the DATA_DIR path.")
else:
    print(f"\nExample track file: {track_files[0]}")
    print(f"Example meta file: {meta_files[0]}")

Found 60 track files.
Found 60 meta files.

Example track file: /kaggle/input/highd-dataset-v1-0/data/01_tracks.csv
Example meta file: /kaggle/input/highd-dataset-v1-0/data/01_tracksMeta.csv


# Define Helper Function: Numerical Features

In [8]:
def extract_numerical_features(df_tracks, df_meta, frame_rate=25):
    """
    Performs Step 1: Extracts all required numerical features from a single recording.
    """
    
    # --- 1. Merge metadata ---
    df = pd.merge(
        df_tracks, 
        df_meta[['id', 'drivingDirection', 'class']], 
        on='id'
    )
    
    # Process both directions. We'll handle lane change logic based on direction.
    df = df[df['drivingDirection'].isin([1, 2])].copy()

    # --- 2. Extract "Target Vehicle Intention" ---
    # We infer intention by looking at the laneId N frames in the future.
    # 3-second window (3s * 25 fps = 75 frames)
    
    df.sort_values(by=['id', 'frame'], inplace=True)
    df['future_laneId'] = df.groupby('id')['laneId'].shift(-75)
    
    def get_intention(row):
        if pd.isna(row['future_laneId']) or row['future_laneId'] == row['laneId']:
            return 'Lane Keeping'
        
        # DrivingDirection 1: Upper lanes, laneId *decreases* for a left change
        if row['drivingDirection'] == 1:
            if row['future_laneId'] < row['laneId']:
                return 'Lane Change Left'
            else:
                return 'Lane Change Right'
        # DrivingDirection 2: Lower lanes, laneId *increases* for a left change
        elif row['drivingDirection'] == 2:
            if row['future_laneId'] > row['laneId']:
                return 'Lane Change Left'
            else:
                return 'Lane Change Right'
        return 'Lane Keeping'

    df['intention'] = df.apply(get_intention, axis=1)
    
    # --- 3. Extract TTC (Time-to-Collision) ---
    vehicle_data = df[['frame', 'id', 'x', 'y', 'xVelocity', 'width', 'height']]
    neighbor_cols = ['leftPrecedingId', 'leftFollowingId', 'rightPrecedingId', 'rightFollowingId']
    
    for col in neighbor_cols:
        # Merge to get neighbor's data
        merged_df = pd.merge(
            df, 
            vehicle_data, 
            left_on=['frame', col], 
            right_on=['frame', 'id'], 
            suffixes=('_ego', '_neighbor'),
            how='left'
        )
        
        ttc_series = np.inf
        if 'Preceding' in col:
            dx = merged_df['x_neighbor'] - merged_df['x_ego'] - merged_df['width_ego']
            dv = merged_df['xVelocity_ego'] - merged_df['xVelocity_neighbor']
        elif 'Following' in col:
            dx = merged_df['x_ego'] - merged_df['x_neighbor'] - merged_df['width_neighbor']
            dv = merged_df['xVelocity_neighbor'] - merged_df['xVelocity_ego']
        
        ttc_series = dx / dv
        ttc_series[dv <= 0] = np.inf
        ttc_series[ttc_series < 0] = np.inf
        
        new_col_name = f'ttc_{col.replace("Id", "")}'
        df[new_col_name] = ttc_series

    # Replace infs/NaNs with 100
    df.replace([np.inf, -np.inf], 100, inplace=True)
    ttc_cols_to_fill = ['ttc'] + [f'ttc_{col.replace("Id", "")}' for col in neighbor_cols]
    df[ttc_cols_to_fill] = df[ttc_cols_to_fill].fillna(100)

    # Define the final feature columns
    numerical_features = [
        'id', 'frame', 'yVelocity', 'yAcceleration', 'intention',
        'ttc', # TTC with in-lane preceding
    ] + [f'ttc_{col.replace("Id", "")}' for col in neighbor_cols]
    
    return df[numerical_features]

def transform_to_linguistic(df, global_stats):
    """
    Performs Step 2: Converts numerical features to linguistic categories
    using pre-calculated global statistics.
    """
    linguistic_df = df.copy()
    
    # --- 1. Lateral Velocity & Acceleration (Global Normal Distribution) ---
    k = 0.5 # The 0.5 std dev threshold
    
    # Categories for Lateral Velocity
    vel_bins = [
        -np.inf, 
        global_stats['lat_vel_mean'] - k * global_stats['lat_vel_std'], 
        global_stats['lat_vel_mean'] + k * global_stats['lat_vel_std'], 
        np.inf
    ]
    vel_labels = ['movingLeft', 'movingStraight', 'movingRight']
    linguistic_df['latVelocity_ling'] = pd.cut(df['yVelocity'], bins=vel_bins, labels=vel_labels, right=False)
    
    # Categories for Lateral Acceleration
    acc_bins = [
        -np.inf, 
        global_stats['lat_acc_mean'] - k * global_stats['lat_acc_std'], 
        global_stats['lat_acc_mean'] + k * global_stats['lat_acc_std'], 
        np.inf
    ]
    acc_labels = ['acceleratingLeft', 'zeroLateralAcceleration', 'acceleratingRight']
    linguistic_df['latAcceleration_ling'] = pd.cut(df['yAcceleration'], bins=acc_bins, labels=acc_labels, right=False)

    # --- 2. TTC (Time-to-Collision) (Thresholds from Literature) ---
    # !!! ASSUMPTION !!!
    # You MUST replace these thresholds with the ones from your cited papers.
    # [0s, 2s] -> HighRisk, (2s, 4s] -> Alert, (4s, 100s] -> Safe
    ttc_bins = [0, 2, 4, 100.1] # 100.1 to include the 100s
    ttc_labels = ['highRisk', 'alert', 'safe']
    
    ttc_cols = [col for col in df.columns if 'ttc' in col]
    for col in ttc_cols:
        linguistic_df[f'{col}_ling'] = pd.cut(df[col], bins=ttc_bins, labels=ttc_labels, right=True)

    # --- 3. Intention ---
    intention_map = {
        'Lane Keeping': 'isMovingStraight',
        'Lane Change Left': 'isChangingLaneLeft',
        'Lane Change Right': 'isChangingLaneRight'
    }
    linguistic_df['intention_ling'] = linguistic_df['intention'].map(intention_map).fillna('isMovingStraight')

    # Select only the new linguistic columns + identifiers
    final_cols = ['id', 'frame'] + [col for col in linguistic_df.columns if '_ling' in col]
    return linguistic_df[final_cols]

# Numerical Feature Extraction

In [9]:
print("--- Starting Pre-computation of Global Stats ---")
print("This may take a minute...")

# We only need these two columns
lat_vel_data = []
lat_acc_data = []

for tracks_file in tqdm(track_files, desc="Calculating Stats"):
    try:
        df = pd.read_csv(tracks_file, usecols=['yVelocity', 'yAcceleration'])
        lat_vel_data.append(df['yVelocity'])
        lat_acc_data.append(df['yAcceleration'])
    except Exception as e:
        print(f"Error reading {tracks_file}: {e}")

# Concatenate all data into two large series
all_lat_vel = pd.concat(lat_vel_data, ignore_index=True)
all_lat_acc = pd.concat(lat_acc_data, ignore_index=True)

# Calculate global stats
global_stats = {
    'lat_vel_mean': all_lat_vel.mean(),
    'lat_vel_std': all_lat_vel.std(),
    'lat_acc_mean': all_lat_acc.mean(),
    'lat_acc_std': all_lat_acc.std(),
}

print("\n--- Global Statistics Calculated ---")
print(f"Lateral Velocity (yVelocity): mean={global_stats['lat_vel_mean']:.4f}, std={global_stats['lat_vel_std']:.4f}")
print(f"Lateral Accel (yAcceleration): mean={global_stats['lat_acc_mean']:.4f}, std={global_stats['lat_acc_std']:.4f}")

--- Starting Pre-computation of Global Stats ---
This may take a minute...


Calculating Stats: 100%|██████████| 60/60 [00:57<00:00,  1.04it/s]



--- Global Statistics Calculated ---
Lateral Velocity (yVelocity): mean=0.0079, std=0.1741
Lateral Accel (yAcceleration): mean=-0.0029, std=0.0762


# Numerical Features and Linguistic Transformation

In [10]:
def process_recording(track_file, meta_file, global_stats):
    """
    Runs the full pipeline for a single recording ID.
    Returns both the numerical and linguistic dataframes.
    """
    try:
        # Get recordingId from filename
        recording_id = int(os.path.basename(track_file).split('_')[0])
        
        # Load data
        df_tracks = pd.read_csv(track_file)
        df_meta = pd.read_csv(meta_file)
        
        # --- Step 1: Get numerical features ---
        numerical_df = extract_numerical_features(df_tracks, df_meta)
        numerical_df['recordingId'] = recording_id
        
        # --- Step 2: Get linguistic features ---
        linguistic_df = transform_to_linguistic(numerical_df, global_stats)
        linguistic_df['recordingId'] = recording_id
        
        return numerical_df, linguistic_df
        
    except Exception as e:
        print(f"Error processing {track_file}: {e}")
        return None, None

# --- Main Execution ---
all_numerical_data = []
all_linguistic_data = []

print(f"\n--- Starting Main Processing for {len(track_files)} Recordings ---")

for track_file, meta_file in tqdm(zip(track_files, meta_files), total=len(track_files), desc="Processing Recordings"):
    num_df, ling_df = process_recording(track_file, meta_file, global_stats)
    if num_df is not None:
        all_numerical_data.append(num_df)
        all_linguistic_data.append(ling_df)

print("\n--- Processing Complete ---")

# Combine all results into two final dataframes
print("Concatenating dataframes...")
final_numerical_df = pd.concat(all_numerical_data, ignore_index=True)
final_linguistic_df = pd.concat(all_linguistic_data, ignore_index=True)

print("\n--- Final Numerical Dataset (for Deep Learning) ---")
print(f"Total rows: {len(final_numerical_df)}")
print(final_numerical_df.head())

print("\n--- Final Linguistic Dataset (for Knowledge Graph) ---")
print(f"Total rows: {len(final_linguistic_df)}")
print(final_linguistic_df.head())


--- Starting Main Processing for 60 Recordings ---


Processing Recordings: 100%|██████████| 60/60 [08:46<00:00,  8.78s/it]



--- Processing Complete ---
Concatenating dataframes...

--- Final Numerical Dataset (for Deep Learning) ---
Total rows: 39725708
   id  frame  yVelocity  yAcceleration     intention  ttc  ttc_leftPreceding  \
0   1      1       0.00            0.0  Lane Keeping  0.0              100.0   
1   1      2       0.00            0.0  Lane Keeping  0.0              100.0   
2   1      3       0.00            0.0  Lane Keeping  0.0              100.0   
3   1      4       0.00            0.0  Lane Keeping  0.0              100.0   
4   1      5       0.01            0.0  Lane Keeping  0.0              100.0   

   ttc_leftFollowing  ttc_rightPreceding  ttc_rightFollowing  recordingId  
0              100.0               100.0               100.0            1  
1              100.0               100.0               100.0            1  
2              100.0               100.0               100.0            1  
3              100.0               100.0               100.0            1  
4       

# Save Numerical Features and Linguistic Transformation

In [11]:
# Define output paths
NUMERICAL_OUTPUT_PATH = '/kaggle/working/numerical_dl_dataset.csv'
LINGUISTIC_OUTPUT_PATH = '/kaggle/working/linguistic_kg_dataset.csv'

# Save the files
print(f"Saving numerical dataset to {NUMERICAL_OUTPUT_PATH}...")
final_numerical_df.to_csv(NUMERICAL_OUTPUT_PATH, index=False)

print(f"Saving linguistic dataset to {LINGUISTIC_OUTPUT_PATH}...")
final_linguistic_df.to_csv(LINGUISTIC_OUTPUT_PATH, index=False)

print("--- All files saved successfully. ---")

Saving numerical dataset to /kaggle/working/numerical_dl_dataset.csv...
Saving linguistic dataset to /kaggle/working/linguistic_kg_dataset.csv...
--- All files saved successfully. ---


In [ ]:
from IPython.display import FileLink, display

# Define the paths
NUMERICAL_OUTPUT_PATH = '/kaggle/working/numerical_dl_dataset.csv'
LINGUISTIC_OUTPUT_PATH = '/kaggle/working/linguistic_kg_dataset.csv'

print("Click the links below to download your files:")

# Create and display the download links
display(FileLink(NUMERICAL_OUTPUT_PATH))
display(FileLink(LINGUISTIC_OUTPUT_PATH))